In [2]:
import os
import requests
import json
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_experimental.sql import SQLDatabaseSequentialChain
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from typing_extensions import Annotated, TypedDict
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
# from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI
import psycopg2
import time 
import yaml
import json 
from langchain_community.agent_toolkits import JsonToolkit, create_json_agent
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI
from langchain.tools import Tool

### Add Thingsboard details

In [3]:
THINGSBOARD_URL = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"
DASHBOARD_ID = "http://localhost:8080/tenants"

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "thingsboard"
DB_USER = "thingsboard"
DB_PASSWORD = "postgres"


THINGSBOARD_HOST = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"

# Authenticate and get the JWT token
auth_url = f'{THINGSBOARD_HOST}/api/auth/login'
auth_payload = {'username': USERNAME, 'password': PASSWORD}
auth_response = requests.post(auth_url, json=auth_payload)
auth_response.raise_for_status()
jwt_token = auth_response.json()['token']

In [4]:
# Load environment variables
load_dotenv()
openai_api_key = os.getenv('OPENAI_PROJECT_API_KEY')

In [78]:
# Load the device metadata
import json 
with open("farm_model_small_v2.json", "r") as file:
    data = json.load(file)

In [6]:
# data['farm']['fields'][0]

In [7]:
def get_farm_details(a: int) -> str:
    """Return Farm field details as a JSON string

    Args:
        a: Field Index
    """
    try:
        field_data = data['farm']['fields'][a]
        return json.dumps(field_data)
    except IndexError:
        return json.dumps({"error": f"Field index {a} is out of bounds."})
    except KeyError:
        return json.dumps({"error": "The 'farm' or 'fields' key was not found in the data."})
    except Exception as e:
        return json.dumps({"error": f"An error occurred: {e}"})

In [8]:
field_index = 0
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F001": {"name": "North Field", "crop": "Maize", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0100"], "field_air_humidity": ["HUM-0100"], "soil_conductivity": ["COND-0100"], "moisture_content": ["MOIST-0100"], "plant_health": ["CAM-0100"]}, "actuator_list": {"pumps": ["PUMP-0100", "PUMP-0101"], "water_valves": ["WV-0100"], "fertilizer_dispensers": ["FD-0100"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0100", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0100", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0100", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [9]:
field_index = 6# Assuming there are fewer than 6 fields
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F007": {"name": "West Field", "crop": "Coffee", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0700"], "field_air_humidity": ["HUM-0700"], "soil_conductivity": ["COND-0700"], "moisture_content": ["MOIST-0700"], "plant_health": ["CAM-0700"]}, "actuator_list": {"pumps": ["PUMP-0700", "PUMP-0701"], "water_valves": ["WV-0700"], "fertilizer_dispensers": ["FD-0700"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0700", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0700", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0700", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [10]:
# create tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_farm_details",
            "description": "Return details about a specific farm field as a JSON object.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "integer",
                        "description": "The index of the field to retrieve details for (0-based).",
                    },
                },
                "required": ["a"],
            },
        },
    }
]

In [11]:
import openai 
# client = openai.OpenAI(api_key=openai_api_key)
client = openai.OpenAI(api_key=openai_api_key)
llm_model = "gpt-3.5-turbo"
# client = ChatOpenAI(api_key=openai_api_key, temperature = 0.0, model=llm_model)
# client = openai.OpenAI(api_key=openai_api_key,model=llm_model)

In [12]:
def run_conversation(query):
    """Runs a conversation with the LLM that can call the get_farm_details tool."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant that identifies relevant devices (sensor_list) mentioned in the user's request and returns them as a JSON array of their IDs e.g. If no specific devices are mentioned, return an empty JSON array."
        "F001 is North Field,"
        "F002 is Northeast Field"
        "F003 is East Field"
        "F004 is Southeast Field"
        "F005 is South Field"
        "F006 is Southwest Field"
        "F007 is West Field"
        "F008 is Northwest Field"
        "F009 is Central Field"
        "Note: if the right Field is not provided return an empty json"
        },
     
        {"role": "user", "content": query}
    ]

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        tools=tools,
        tool_choice="auto", 
    )

    response_message = response.choices[0].message

    if response_message.tool_calls:
        print("LLM initiated a tool call:")
        print(response_message.tool_calls)

        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        function_to_call = globals()[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(**function_args)

        print(f"Calling function '{function_name}' with arguments: {function_args}")
        print(f"Function returned: {function_response}")

        messages.append(response_message)
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            }
        )
        second_response = client.chat.completions.create(
            # model="gpt-3.5-turbo-0613",
            model="gpt-3.5-turbo",
            messages=messages,
        )
        
        return second_response.choices[0].message.content
    else:
        # If no tool call, the LLM might be directly answering based on the system prompt
        try:
            # Attempt to parse the response as JSON (assuming it followed the system prompt)
            return json.loads(response_message.content)
        except (json.JSONDecodeError, TypeError):
            # If it's not valid JSON, return the raw content
            return response_message.content

In [13]:
user_query_farm = "Tell me the details of the sensors in the south  field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_JiE13StaSDVtnhhBZ7dGTCMO', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [14]:
user_query_farm = "Tell me the details of the sensors in the east field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_zQcttS1qAIgt5PXHnD26hu6F', function=Function(arguments='{"a":2}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 2}
Function returned: {"F003": {"name": "East Field", "crop": "Sorghum", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0300"], "field_air_humidity": ["HUM-0300"], "soil_conductivity": ["COND-0300"], "moisture_content": ["MOIST-0300"], "plant_health": ["CAM-0300"]}, "actuator_list": {"pumps": ["PUMP-0300", "PUMP-0301"], "water_valves": ["WV-0300"], "fertilizer_dispensers": ["FD-0300"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0300", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_

In [15]:
user_query_farm = "Tell me the details of the sensors in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_ru4y8vfysbsSnLAEC7ci0bZl', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [16]:
user_query_farm = "Tell me the details of the temperature sensor in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_TzepdgaJsDFEb4RYIaY0SDCu', function=Function(arguments='{"a":5}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 5}
Function returned: {"F006": {"name": "Southwest Field", "crop": "Soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0600"], "field_air_humidity": ["HUM-0600"], "soil_conductivity": ["COND-0600"], "moisture_content": ["MOIST-0600"], "plant_health": ["CAM-0600"]}, "actuator_list": {"pumps": ["PUMP-0600", "PUMP-0601"], "water_valves": ["WV-0600"], "fertilizer_dispensers": ["FD-0600"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0600", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field

In [17]:
user_query_farm = "Tell me the details of the temperature sensor in the give South-West field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_GkjSwUHe4IBY8h5acbHujdkL', function=Function(arguments='{"a":5}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 5}
Function returned: {"F006": {"name": "Southwest Field", "crop": "Soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0600"], "field_air_humidity": ["HUM-0600"], "soil_conductivity": ["COND-0600"], "moisture_content": ["MOIST-0600"], "plant_health": ["CAM-0600"]}, "actuator_list": {"pumps": ["PUMP-0600", "PUMP-0601"], "water_valves": ["WV-0600"], "fertilizer_dispensers": ["FD-0600"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0600", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field

In [18]:
sensor_ids = json.loads(response_farm)["sensor_list"]
sensor_ids

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# 2. Get device ID by name
def get_device_id_by_name(device_name, token):
    headers = {
        "Content-Type": "application/json",
        "X-Authorization": f"Bearer {token}"
    }
    url = f"{THINGSBOARD_URL}/api/tenant/devices?deviceName={device_name}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    device = response.json()
    return device['id']['id'] if device else None

In [ ]:
def get_device_keys(jwt_token, device_id):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/keys/timeseries"
    headers = {
        "X-Authorization": f"Bearer {jwt_token}"
    }

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()  # Returns a list of key names
    else:
        return {
            "error": f"Failed to fetch keys: {response.status_code}",
            "details": response.text
        }

In [ ]:
# change the device name 
device_name = sensor_ids[0] #"HUM-0100"
sensor_id = get_device_id_by_name(device_name, jwt_token)
device_id = sensor_id
sensor_id

'e82fc1b0-1221-11f0-a236-5f0808fc8cdf'

In [ ]:
device_key = get_device_keys(jwt_token, sensor_id)
device_key = device_key[-1] # ['deviceId', 'unit', 'relative_humidity']
device_key 

'temp'

# Get data for the last 24 hours

In [ ]:
# Last 24 hours timestamps
end_ts = int(time.time() * 1000)  
start_ts = end_ts - (340 * 60 * 60 * 1000)  # 24 hours ago

# keys = "temperature"  
keys = device_key


def get_historical_data(jwt_token, device_id, start_ts, end_ts, keys):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/values/timeseries"
    params = {"keys": keys, "startTs": start_ts, "endTs": end_ts, "limit": 100}
    headers = {"X-Authorization": f"Bearer {jwt_token}"}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json() if response.status_code == 200 else {"error": "Failed to fetch telemetry data"}

response = get_historical_data(jwt_token, sensor_id, start_ts, end_ts, keys)
response

{'temp': [{'ts': 1743869011403, 'value': '27.320302266151312'},
  {'ts': 1743868709415, 'value': '31.306305009158017'},
  {'ts': 1743868407223, 'value': '33.48834158762985'},
  {'ts': 1743868105251, 'value': '34.10189227675727'},
  {'ts': 1743867803070, 'value': '28.534077498339663'},
  {'ts': 1743867501220, 'value': '27.817652246517067'},
  {'ts': 1743867199515, 'value': '32.49147749209336'},
  {'ts': 1743866897596, 'value': '28.854327077358164'},
  {'ts': 1743866595362, 'value': '28.009212122493373'},
  {'ts': 1743866293508, 'value': '31.45880622072238'},
  {'ts': 1743865991358, 'value': '33.49596248476476'},
  {'ts': 1743864788511, 'value': '25.02549472745727'}]}

In [ ]:
telemetry_data = response

In [28]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch

In [109]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("document.txt")

documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(api_key=openai_api_key)
vectorstore = FAISS.from_documents(texts, embeddings)


In [110]:
retriever = vectorstore.as_retriever()

In [111]:
# Initialize your language model
llm = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")

# Create the RetrievalQA chain
retrieval_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }

)

In [112]:
from langchain.evaluation.qa import QAGenerateChain
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo"))


In [113]:
data

{'farm': {'fields': [{'F001': {'name': 'North Field',
     'crop': 'Maize',
     'area': '62.5 acres',
     'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
      'east': {'lat': 35.6789, 'long': -98.1234},
      'south': {'lat': 35.6789, 'long': -98.1234},
      'west': {'lat': 35.6789, 'long': -98.1234}},
     'sensor_list': {'soil_temperature': ['TEMP-0100'],
      'field_air_humidity': ['HUM-0100'],
      'soil_conductivity': ['COND-0100'],
      'moisture_content': ['MOIST-0100'],
      'plant_health': ['CAM-0100']},
     'actuator_list': {'pumps': ['PUMP-0100', 'PUMP-0101'],
      'water_valves': ['WV-0100'],
      'fertilizer_dispensers': ['FD-0100']},
     'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0100',
        'gps': {'lat': 35.6801, 'long': -98.1201},
        'status': 'transmitting',
        'unit': 'celsius'}],
      'field_air_humidity': [{'sensor_id': 'HUM-0100',
        'gps': {'lat': 35.6802, 'long': -98.1202},
        'status': 'transmitting',


In [114]:
fields = data['farm']['fields']

In [115]:
len(fields)

9

In [116]:
from langchain.chat_models import ChatOpenAI

# Initialize the language model
chat_model = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")


# Construct sensor info string for the LLM prompt
field_sensor_details = ""
for field in fields:
    for field_id, field_info in field.items():
        crop = field_info['crop']
        field_sensor_details += f"Field ID: {field_id}, Crop: {crop}\n"

        for sensor_category, sensors in field_info['sensors'].items():
            for sensor in sensors:
                sensor_id = sensor.get('sensor_id') or sensor.get('camera_id')
                sensor_type = sensor_category.replace('_', ' ').capitalize()
                field_sensor_details += f"  - Sensor ID: {sensor_id}, Type: {sensor_type}\n"
        field_sensor_details += "\n"

# Compose prompt for generating question-answer pairs
instruction = f"""
Below is a list of fields and their associated sensors in a smart farming system:

{field_sensor_details}

Using the above information, create 20 unique question-answer pairs. Focus on:
1. Retrieving the sensor ID based on sensor type and field.

Format each entry as a dictionary with 'query' and 'answer' keys:
{{"query": "...", "answer": "..."}}
"""

# Request generation from the LLM
qa_pairs = chat_model.predict(instruction)
print(qa_pairs)



1. {"query": "What is the sensor ID for soil temperature in Field F003?", "answer": "Sensor ID: TEMP-0300"}
2. {"query": "Which sensor ID corresponds to moisture content in Field F008?", "answer": "Sensor ID: MOIST-0800"}
3. {"query": "In Field F005, what is the sensor ID for field air humidity?", "answer": "Sensor ID: HUM-0500"}
4. {"query": "What is the sensor ID for soil conductivity in Field F002?", "answer": "Sensor ID: COND-0200"}
5. {"query": "Which sensor ID is associated with plant health cameras in Field F006?", "answer": "Sensor ID: CAM-0600"}
6. {"query": "What is the sensor ID for field air humidity in Field F004?", "answer": "Sensor ID: HUM-0400"}
7. {"query": "Which sensor ID corresponds to soil temperature in Field F009?", "answer": "Sensor ID: TEMP-0900"}
8. {"query": "In Field F001, what is the sensor ID for moisture content?", "answer": "Sensor ID: MOIST-0100"}
9. {"query": "What is the sensor ID for soil temperature in Field F007?", "answer": "Sensor ID: TEMP-0700"

In [83]:
# qa_pairs['query']

In [117]:
import re

# Extract JSON-like dictionaries using regex
json_like_items = re.findall(r'\{.*?\}', qa_pairs, re.DOTALL)

# Convert them into actual dictionaries
qa_pairs = [json.loads(item) for item in json_like_items]

In [118]:
qa_pairs

[{'query': 'What is the sensor ID for soil temperature in Field F003?',
  'answer': 'Sensor ID: TEMP-0300'},
 {'query': 'Which sensor ID corresponds to moisture content in Field F008?',
  'answer': 'Sensor ID: MOIST-0800'},
 {'query': 'In Field F005, what is the sensor ID for field air humidity?',
  'answer': 'Sensor ID: HUM-0500'},
 {'query': 'What is the sensor ID for soil conductivity in Field F002?',
  'answer': 'Sensor ID: COND-0200'},
 {'query': 'Which sensor ID is associated with plant health cameras in Field F006?',
  'answer': 'Sensor ID: CAM-0600'},
 {'query': 'What is the sensor ID for field air humidity in Field F004?',
  'answer': 'Sensor ID: HUM-0400'},
 {'query': 'Which sensor ID corresponds to soil temperature in Field F009?',
  'answer': 'Sensor ID: TEMP-0900'},
 {'query': 'In Field F001, what is the sensor ID for moisture content?',
  'answer': 'Sensor ID: MOIST-0100'},
 {'query': 'What is the sensor ID for soil temperature in Field F007?',
  'answer': 'Sensor ID: TEM

In [119]:
qa_pairs[0]

{'query': 'What is the sensor ID for soil temperature in Field F003?',
 'answer': 'Sensor ID: TEMP-0300'}

In [120]:
retrieval_qa.run(qa_pairs[0]["query"])

'The sensor ID for soil temperature in Field F003 is TEMP-0300.'

In [122]:
predictions = retrieval_qa.run(qa_pairs[0]["query"])
predictions

'The sensor ID for soil temperature in Field F003 is TEMP-0100.'

In [123]:
from langchain.evaluation.qa import QAEvalChain

In [124]:
eval_chain = QAEvalChain.from_llm(llm)

In [125]:
qa_pairs

[{'query': 'What is the sensor ID for soil temperature in Field F003?',
  'answer': 'Sensor ID: TEMP-0300'},
 {'query': 'Which sensor ID corresponds to moisture content in Field F008?',
  'answer': 'Sensor ID: MOIST-0800'},
 {'query': 'In Field F005, what is the sensor ID for field air humidity?',
  'answer': 'Sensor ID: HUM-0500'},
 {'query': 'What is the sensor ID for soil conductivity in Field F002?',
  'answer': 'Sensor ID: COND-0200'},
 {'query': 'Which sensor ID is associated with plant health cameras in Field F006?',
  'answer': 'Sensor ID: CAM-0600'},
 {'query': 'What is the sensor ID for field air humidity in Field F004?',
  'answer': 'Sensor ID: HUM-0400'},
 {'query': 'Which sensor ID corresponds to soil temperature in Field F009?',
  'answer': 'Sensor ID: TEMP-0900'},
 {'query': 'In Field F001, what is the sensor ID for moisture content?',
  'answer': 'Sensor ID: MOIST-0100'},
 {'query': 'What is the sensor ID for soil temperature in Field F007?',
  'answer': 'Sensor ID: TEM

In [126]:
predictions = retrieval_qa.apply(qa_pairs)

In [103]:
predictions

[{'query': 'What is the sensor ID for soil temperature in Field F003?',
  'answer': 'Sensor ID: TEMP-0300',
  'result': 'The sensor ID for soil temperature in Field F003 is TEMP-0100.'},
 {'query': 'From sensor ID HUM-0500, what crop is being monitored?',
  'answer': 'Crop: Soybean',
  'result': "I don't have information about sensor ID HUM-0500 in the provided context."},
 {'query': 'Which field does sensor ID COND-0400 belong to?',
  'answer': 'Field ID: F004',
  'result': "I don't have information on sensor ID COND-0400 to determine which field it belongs to."},
 {'query': 'What type of sensor monitors plant health in Field F007?',
  'answer': 'Type: Plant health cameras',
  'result': "There is no information provided for Field F007 in the context given. Therefore, I don't have details about the sensors or actuators specific to Field F007."},
 {'query': 'What is the sensor type for sensor ID MOIST-0600?',
  'answer': 'Type: Moisture content',
  'result': "I don't have information ab

In [127]:
graded_outputs = eval_chain.evaluate(qa_pairs, predictions)

In [128]:
graded_outputs

[{'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'GRADE: INCORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [129]:
for i, eg in enumerate(qa_pairs):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()


Example 0:
Question: What is the sensor ID for soil temperature in Field F003?
Real Answer: Sensor ID: TEMP-0300
Predicted Answer: The sensor ID for soil temperature in Field F003 is TEMP-0100.
Predicted Grade: INCORRECT

Example 1:
Question: Which sensor ID corresponds to moisture content in Field F008?
Real Answer: Sensor ID: MOIST-0800
Predicted Answer: I'm sorry, but based on the provided information, there is no Field F008 mentioned in the context. Therefore, I cannot determine which sensor ID corresponds to moisture content in Field F008.
Predicted Grade: CORRECT

Example 2:
Question: In Field F005, what is the sensor ID for field air humidity?
Real Answer: Sensor ID: HUM-0500
Predicted Answer: I don't have any information about Field F005 in the provided context.
Predicted Grade: CORRECT

Example 3:
Question: What is the sensor ID for soil conductivity in Field F002?
Real Answer: Sensor ID: COND-0200
Predicted Answer: The sensor ID for soil conductivity in Field F002 is COND-030